# Clase 186 — CUPED, sequential testing y always-valid p-values

Tres trucos modernos de A/B testing industrial: **CUPED** reduce varianza usando pre-experiment data (~50% menos sample). **Sequential testing** te deja peekear sin inflar el error tipo I.
Requiere: `pip install numpy scipy`.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
n = 10_000
# Pre-experiment covariate (X), correlación ~0.7 con outcome
X_ctrl = rng.normal(10, 3, n)
X_trt  = rng.normal(10, 3, n)
noise  = lambda k: rng.normal(0, 2.1, k)  # ruido idiosincratico
Y_ctrl = X_ctrl + noise(n)             # mean ~ 10
Y_trt  = X_trt + noise(n) + 0.5        # treatment effect = 0.5
print(f'mean ctrl = {Y_ctrl.mean():.3f}  mean trt = {Y_trt.mean():.3f}  diff = {Y_trt.mean()-Y_ctrl.mean():.3f}')

## t-test naive

In [ ]:
t, p = stats.ttest_ind(Y_trt, Y_ctrl)
var_naive = (Y_trt.var(ddof=1) + Y_ctrl.var(ddof=1)) / 2
print(f'Naive: t={t:.3f}  p={p:.4f}  pooled var ~ {var_naive:.3f}')

## CUPED — Controlled-experiment Using Pre-Experiment Data
$Y^{cuped}_i = Y_i - \theta (X_i - \bar X)$ con $\theta = \dfrac{\text{Cov}(Y,X)}{\text{Var}(X)}$.
$\text{Var}(Y^{cuped}) = \text{Var}(Y)(1 - \rho^2)$ — con $\rho=0.7$, reduce ~50% varianza.

In [ ]:
# Estimamos theta sobre TODO el pool (no por grupo) para no introducir bias
Y_all = np.concatenate([Y_ctrl, Y_trt])
X_all = np.concatenate([X_ctrl, X_trt])
theta = np.cov(Y_all, X_all, ddof=1)[0, 1] / X_all.var(ddof=1)
X_bar = X_all.mean()
Yc_cuped = Y_ctrl - theta * (X_ctrl - X_bar)
Yt_cuped = Y_trt  - theta * (X_trt  - X_bar)

t_c, p_c = stats.ttest_ind(Yt_cuped, Yc_cuped)
var_cuped = (Yt_cuped.var(ddof=1) + Yc_cuped.var(ddof=1)) / 2
print(f'theta = {theta:.3f}')
print(f'CUPED: t={t_c:.3f}  p={p_c:.4e}  pooled var ~ {var_cuped:.3f}')
print(f'Reduccion de varianza: {1 - var_cuped/var_naive:.1%}  (esperado ~50% con rho=0.7)')

## El peligro del *peeking* sin corrección
Simulamos 1000 experimentos NULOS (sin efecto), peekeando 10 veces. Sin corrección, P(rechazar H0) >> 5%.

In [ ]:
rng2 = np.random.default_rng(42)
n_sims = 1000
n_max = 2000
n_peeks = 10
peek_at = np.linspace(200, n_max, n_peeks).astype(int)

rejects_naive = 0
for _ in range(n_sims):
    a = rng2.normal(0, 1, n_max)
    b = rng2.normal(0, 1, n_max)  # SIN efecto
    for k in peek_at:
        _, p = stats.ttest_ind(a[:k], b[:k])
        if p < 0.05:
            rejects_naive += 1
            break
print(f'Type-I error peekeando sin corregir: {rejects_naive/n_sims:.3f}  (debería ser 0.05)')

## Always-valid p-values — corrección Bonferroni para peeks
Si peekeás K veces, usá $\alpha/K$. Conservador pero válido.

In [ ]:
alpha_corr = 0.05 / n_peeks
rng3 = np.random.default_rng(42)
rejects_corr = 0
for _ in range(n_sims):
    a = rng3.normal(0, 1, n_max)
    b = rng3.normal(0, 1, n_max)
    for k in peek_at:
        _, p = stats.ttest_ind(a[:k], b[:k])
        if p < alpha_corr:
            rejects_corr += 1
            break
print(f'Type-I con Bonferroni K={n_peeks} (alpha={alpha_corr:.4f}): {rejects_corr/n_sims:.3f}')

## mSPRT-like: Mixture Sequential Probability Ratio Test
La idea: bajo H0, el likelihood ratio mezclado contra un prior $N(0,\tau^2)$ es martingala. P-value siempre válido en cualquier momento.

In [ ]:
def msprt_pvalue(diffs, sigma, tau=0.1):
    """Always-valid p-value para diferencia de medias bajo H0=0.
    diffs: muestras Y_trt - Y_ctrl pareadas (o pseudo-pareadas por bloque).
    sigma: SD conocida o estimada.
    tau: SD del prior sobre el efecto."""
    n = len(diffs)
    s = diffs.sum()
    # log Bayes factor mezclado
    var_tot = sigma**2 + n*tau**2
    log_bf = 0.5*np.log(sigma**2 / var_tot) + 0.5 * (s*tau)**2 / (sigma**2 * var_tot)
    bf = np.exp(log_bf)
    return min(1.0, 1.0/bf)

rng4 = np.random.default_rng(42)
# Bajo H0: 1000 sims, peekear 10 veces, contar rechazos
rej_msprt = 0
for _ in range(n_sims):
    d = rng4.normal(0, 1, n_max)
    for k in peek_at:
        p = msprt_pvalue(d[:k], sigma=1.0, tau=0.2)
        if p < 0.05:
            rej_msprt += 1
            break
print(f'Type-I con mSPRT (peekeando libre): {rej_msprt/n_sims:.3f}  (target <=0.05)')

## Comparativo final

In [ ]:
print('Tipo I esperado: 0.05')
print(f'Naive peeking      : {rejects_naive/n_sims:.3f}  (inflado)')
print(f'Bonferroni K-peeks : {rejects_corr/n_sims:.3f}  (controlado, conservador)')
print(f'mSPRT always-valid : {rej_msprt/n_sims:.3f}    (controlado, less conservative)')

## Takeaways
1. **CUPED** reduce ~50% varianza con $\rho=0.7$ → mismos resultados con la mitad del tráfico.
2. Peekear sin corrección **infla el type-I error** del 5% al ~20-30%.
3. Bonferroni sobre K peeks es válido pero conservador.
4. **mSPRT / always-valid p-values** te dejan monitorear continuamente sin inflar error — preferido en industria (Optimizely, Netflix).